In [1]:
# =============================================================================
# RiceCast TraderEdge — Model Training & Backtest
# =============================================================================
# Prerequisites:
#   - data/processed/master_df.csv  ✓ (from 01_master_df.py)
#   - data/processed/train_df.csv   ✓ (from 01_master_df.py)
#   - data/processed/holdout_df.csv ✓ (from 01_master_df.py)
#
# Output:
#   - models/prophet_model.pkl       ← used by Azure Function
#   - models/model_card.md
#   - models/backtest_chart.png      ← the hero demo slide
# =============================================================================

In [ ]:
# CELL 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
import json
import os
import warnings
warnings.filterwarnings("ignore")
 
from prophet import Prophet
from sklearn.metrics import mean_absolute_error
 
os.makedirs("models", exist_ok=True)
 
print("Libraries loaded ✓")

In [ ]:
# Load data
 
master  = pd.read_csv("data/processed/master_df.csv",   parse_dates=["ds"])
train   = pd.read_csv("data/processed/train_df.csv",    parse_dates=["ds"])
holdout = pd.read_csv("data/processed/holdout_df.csv",  parse_dates=["ds"])
 
FEATURE_COLS = [
    "production_dev_pct",
    "price_mom_3m",
    "price_accel",
    "harvest_window",
    "rainfall_dev_pct",
]
 
print(f"master  : {len(master)} rows  ({master['ds'].min().date()} → {master['ds'].max().date()})")
print(f"train   : {len(train)} rows   ({train['ds'].min().date()} → {train['ds'].max().date()})")
print(f"holdout : {len(holdout)} rows  ({holdout['ds'].min().date() if len(holdout)>0 else 'n/a'} → {holdout['ds'].max().date() if len(holdout)>0 else 'n/a'})")
print(f"\nFeatures: {FEATURE_COLS}")

In [ ]:
# Build and fit Prophet model
# seasonality_mode = 'additive' based on EDA finding:
#   seasonal factor range = 0.041 < 0.05 threshold
#   → additive models fixed IDR/kg seasonal swings (appropriate for this data)
 
m = Prophet(
    seasonality_mode="additive",        # ← changed from multiplicative based on EDA
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,       # conservative — avoid overfitting
    seasonality_prior_scale=10,         # default, let seasonality speak
    interval_width=0.80                 # 80% CI — honest uncertainty
)
 
# Add regressors with documented prior scales
# prior_scale controls how much each regressor can influence the forecast
# Higher = more flexible, Lower = more conservative
m.add_regressor("production_dev_pct", prior_scale=0.5,  standardize=True)
m.add_regressor("price_mom_3m",       prior_scale=0.5,  standardize=True)
m.add_regressor("price_accel",        prior_scale=0.3,  standardize=True)
m.add_regressor("harvest_window",     prior_scale=0.4,  standardize=False)
m.add_regressor("rainfall_dev_pct",   prior_scale=0.02, standardize=True)
 
print("Fitting Prophet model on training data...")
print(f"  Training rows : {len(train)}")
print(f"  Seasonality   : additive")
print(f"  Regressors    : {FEATURE_COLS}")
 
m.fit(train)
print("\n✓ Model fitted successfully")

In [ ]:
# %% ─── CELL 4: Generate forecast on full data range ─────────────────────────
# We forecast across the entire master_df date range (train + holdout)
# so we can compare predictions vs actuals for the backtest.
 
future = m.make_future_dataframe(
    periods=len(holdout) + 6,   # holdout months + 6 months ahead
    freq="MS"                   # month start frequency
)
 
# Merge known feature values into the future dataframe
future = future.merge(
    master[["ds"] + FEATURE_COLS],
    on="ds", how="left"
)
 
# Fill unknown future months (beyond master data) with neutral/seasonal estimates
for col in FEATURE_COLS:
    future[col] = future[col].fillna(0)
 
# Ensure harvest_window is correct for all future dates (not just filled zeros)
future["harvest_window"] = future["ds"].dt.month.isin([3, 4, 5, 7, 8, 9]).astype(int)
 
print(f"Future dataframe: {len(future)} rows")
print(f"  Covers: {future['ds'].min().date()} → {future['ds'].max().date()}")
 
forecast = m.predict(future)
print(f"\nForecast generated: {len(forecast)} rows")
print(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(8).to_string(index=False))

In [ ]:
# %% ─── CELL 5: Plot components ──────────────────────────────────────────────
# Shows trend, seasonality, and each regressor's contribution.
# Useful for explaining the model to judges.
 
fig1 = m.plot(forecast, figsize=(14, 5))
plt.title("Prophet Forecast — Rice Price, Jawa Timur (IDR/kg)", fontsize=13)
plt.ylabel("IDR/kg")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"Rp {x:,.0f}"))
plt.tight_layout()
plt.savefig("models/forecast_plot.png", dpi=130, bbox_inches="tight")
plt.show()
 
fig2 = m.plot_components(forecast)
plt.tight_layout()
plt.savefig("models/components_plot.png", dpi=130, bbox_inches="tight")
plt.show()
 
print("Saved: models/forecast_plot.png")
print("Saved: models/components_plot.png")

In [ ]:
# %% ─── CELL 6: Holdout validation — directional accuracy ───────────────────
# PRIMARY metric: did the model predict the correct direction of price movement?
# We use directional accuracy, not MAE, because:
#   - Our product gives directional signals (GLUT/SHORTAGE), not exact prices
#   - MAE would be misleadingly precise given our data resolution
# A model that says "price will fall" when it actually falls = correct direction.
 
if len(holdout) == 0:
    print("No holdout data yet (all data is pre-2024) — skipping validation")
    print("Backtest against known events (Cell 8) will be used instead.")
    dir_acc = None
    mae = None
    mape = None
else:
    holdout_fc = forecast[forecast["ds"].isin(holdout["ds"])][
        ["ds", "yhat", "yhat_lower", "yhat_upper"]
    ].merge(holdout[["ds", "y"]], on="ds")
 
    holdout_fc = holdout_fc.sort_values("ds").reset_index(drop=True)
 
    # Directional accuracy
    holdout_fc["actual_dir"] = np.sign(holdout_fc["y"].diff())
    holdout_fc["pred_dir"]   = np.sign(holdout_fc["yhat"].diff())
    valid_rows = holdout_fc.dropna(subset=["actual_dir"])
    dir_acc = (valid_rows["actual_dir"] == valid_rows["pred_dir"]).mean()
 
    # MAE and MAPE (secondary)
    mae  = mean_absolute_error(holdout_fc["y"], holdout_fc["yhat"])
    mape = (np.abs(holdout_fc["y"] - holdout_fc["yhat"]) / holdout_fc["y"]).mean() * 100
 
    print("=" * 50)
    print("HOLDOUT VALIDATION RESULTS")
    print("=" * 50)
    print(f"Directional accuracy : {dir_acc:.1%}  {'✓' if dir_acc > 0.55 else '⚠'}")
    print(f"MAE                  : Rp {mae:,.0f}/kg  (reference only)")
    print(f"MAPE                 : {mape:.1f}%          (reference only)")
    print()
    print("Target: directional accuracy > 55%")
    print("Note: MAE/MAPE are shown for reference. Our product outputs")
    print("      directional signals, so directional accuracy is what matters.")
 
    print("\nHoldout detail:")
    print(holdout_fc[["ds","y","yhat","actual_dir","pred_dir"]].to_string(index=False))

In [ ]:
# %% ─── CELL 7: Supply pressure scoring function ─────────────────────────────
# This is the core of TraderEdge — converts Prophet output to GLUT/SHORTAGE signal.
# Mirrors the logic in function/supply_pressure_scorer.py exactly.
 
def compute_supply_pressure(forecast_df: pd.DataFrame, current_price: float) -> dict:
    """
    Takes the 'future-only' slice of Prophet forecast (next N months).
    Returns the supply pressure signal dict.
    """
    if len(forecast_df) == 0 or current_price <= 0:
        return {"dominant_signal": "NEUTRAL", "intensity": "LOW",
                "glut_score": 0, "shortage_score": 0}
 
    last_yhat        = forecast_df["yhat"].iloc[-1]
    price_change_pct = (last_yhat - current_price) / current_price
 
    harvest_near = int(forecast_df["ds"].dt.month.isin([3,4,5,7,8,9]).sum() >= 2)
    lean_near    = int(forecast_df["ds"].dt.month.isin([10,11,12,1,2]).sum() >= 2)
 
    ci_width_pct = float(
        ((forecast_df["yhat_upper"] - forecast_df["yhat_lower"]) / current_price)
        .mean() * 100
    )
 
    # Glut score: price falling + harvest season approaching
    glut_score = max(0.0, -price_change_pct * 300) + (25 if harvest_near else 0)
    glut_score = min(100.0, round(glut_score, 1))
 
    # Shortage score: price rising + lean season approaching
    shortage_score = max(0.0, price_change_pct * 300) + (20 if lean_near else 0)
    shortage_score = min(100.0, round(shortage_score, 1))
 
    dominant = (
        "GLUT"     if glut_score > shortage_score and glut_score >= 20
        else "SHORTAGE" if shortage_score > glut_score and shortage_score >= 20
        else "NEUTRAL"
    )
    intensity = (
        "HIGH"   if max(glut_score, shortage_score) >= 70
        else "MEDIUM" if max(glut_score, shortage_score) >= 40
        else "LOW"
    )
 
    return {
        "dominant_signal":  dominant,
        "intensity":        intensity,
        "glut_score":       int(glut_score),
        "shortage_score":   int(shortage_score),
        "ci_width_pct":     round(ci_width_pct, 1),
        "price_change_pct": round(price_change_pct * 100, 1),
        "forecast_values":  forecast_df["yhat"].round(0).astype(int).tolist(),
        "ci_lower":         forecast_df["yhat_lower"].round(0).astype(int).tolist(),
        "ci_upper":         forecast_df["yhat_upper"].round(0).astype(int).tolist(),
        "forecast_dates":   forecast_df["ds"].dt.strftime("%Y-%m-%d").tolist(),
    }
 
print("Supply pressure scoring function defined ✓")

In [ ]:
# %% ─── CELL 8: Rolling signal generation for backtest ──────────────────────
# Walk forward through history month by month.
# At each month: use model to forecast 3 months ahead, compute supply pressure score.
# This builds a full history of what TraderEdge would have signalled.
 
print("Generating rolling supply pressure signals...")
signal_history = []
 
for i in range(6, len(master)):
    row           = master.iloc[i]
    current_price = row["y"]
    ds_current    = row["ds"]
 
    # Build 3-month ahead future slice
    future_dates = pd.date_range(
        ds_current + pd.DateOffset(months=1),
        periods=3, freq="MS"
    )
    future_3m = pd.DataFrame({"ds": future_dates})
    future_3m = future_3m.merge(
        master[["ds"] + FEATURE_COLS], on="ds", how="left"
    )
    for col in FEATURE_COLS:
        future_3m[col] = future_3m[col].fillna(0)
    future_3m["harvest_window"] = future_3m["ds"].dt.month.isin([3,4,5,7,8,9]).astype(int)
 
    fc_3m = m.predict(future_3m)
    signal = compute_supply_pressure(fc_3m, current_price)
    signal["ds"]           = ds_current
    signal["actual_price"] = current_price
    signal_history.append(signal)
 
signals_df = pd.DataFrame(signal_history)
print(f"✓ Generated {len(signals_df)} monthly signals")
print(f"\nSignal distribution:")
print(signals_df["dominant_signal"].value_counts())
print()
print(signals_df[["ds","dominant_signal","intensity","glut_score","shortage_score"]].tail(12).to_string(index=False))

In [ ]:
# %% ─── CELL 9: BACKTEST — Hero chart ────────────────────────────────────────
# The most important output of this notebook.
# Shows whether TraderEdge would have warned traders BEFORE known events.
# This chart goes on the demo slide.
 
EVENTS = [
    {
        "label":       "Post-Panen Glut Q2 2022",
        "signal_type": "GLUT",
        "start":       "2022-04-01",
        "end":         "2022-06-30",
        "color":       "#EF9F27",
    },
    {
        "label":       "El Niño Shortage Q3 2023",
        "signal_type": "SHORTAGE",
        "start":       "2023-07-01",
        "end":         "2023-10-31",
        "color":       "#E24B4A",
    },
]
 
fig, axes = plt.subplots(3, 1, figsize=(15, 13), sharex=True)
 
# ── Panel 1: Actual price ─────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(master["ds"], master["y"],
         color="#1B3A6B", linewidth=2, zorder=3, label="Actual price")
ax1.set_title("Rice Price — Jawa Timur (IDR/kg)", fontsize=12, fontweight="bold")
ax1.set_ylabel("IDR/kg")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"Rp {x:,.0f}"))
ax1.legend(fontsize=9, loc="upper left")
ax1.grid(axis="y", alpha=0.3)
 
# ── Panel 2: Supply pressure scores ──────────────────────────────────────
ax2 = axes[1]
ax2.fill_between(signals_df["ds"], signals_df["glut_score"],
                 alpha=0.5, color="#EF9F27", label="Glut score", step="mid")
ax2.fill_between(signals_df["ds"], signals_df["shortage_score"],
                 alpha=0.5, color="#E24B4A", label="Shortage score", step="mid")
ax2.axhline(70, color="#374151", linestyle="--", linewidth=0.8, alpha=0.7)
ax2.axhline(40, color="#374151", linestyle=":",  linewidth=0.8, alpha=0.5)
ax2.text(signals_df["ds"].iloc[-1], 71, "HIGH (70)", fontsize=8, color="#374151", va="bottom")
ax2.text(signals_df["ds"].iloc[-1], 41, "MEDIUM (40)", fontsize=8, color="#374151", va="bottom")
ax2.set_title("Supply Pressure Score (0–100)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Score")
ax2.set_ylim(0, 110)
ax2.legend(fontsize=9, loc="upper left")
ax2.grid(axis="y", alpha=0.3)
 
# ── Panel 3: Signal intensity ─────────────────────────────────────────────
ax3 = axes[2]
intensity_val = {"HIGH": 3, "MEDIUM": 2, "LOW": 1}
signal_color  = {"GLUT": "#EF9F27", "SHORTAGE": "#E24B4A", "NEUTRAL": "#9CA3AF"}
 
for _, row in signals_df.iterrows():
    ax3.bar(
        row["ds"],
        intensity_val.get(row["intensity"], 1),
        width=20,
        color=signal_color.get(row["dominant_signal"], "#9CA3AF"),
        alpha=0.85
    )
ax3.set_yticks([1, 2, 3])
ax3.set_yticklabels(["LOW", "MED", "HIGH"])
ax3.set_title("Signal Output (color = type)", fontsize=12, fontweight="bold")
legend_items = [
    mpatches.Patch(color="#EF9F27", label="GLUT"),
    mpatches.Patch(color="#E24B4A", label="SHORTAGE"),
    mpatches.Patch(color="#9CA3AF", label="NEUTRAL"),
]
ax3.legend(handles=legend_items, fontsize=9, loc="upper left")
ax3.set_xlabel("")
ax3.grid(axis="y", alpha=0.3)
 
# ── Shade known event windows on all panels ───────────────────────────────
for event in EVENTS:
    t_start = pd.Timestamp(event["start"])
    t_end   = pd.Timestamp(event["end"])
    if t_start < master["ds"].max():
        for ax in axes:
            ax.axvspan(t_start, t_end,
                       alpha=0.10, color=event["color"],
                       label=f"{event['label']}")
 
# Add event labels only on top panel
for event in EVENTS:
    t_start = pd.Timestamp(event["start"])
    if t_start < master["ds"].max():
        price_at_event = master[master["ds"] >= t_start]["y"].iloc[0]
        axes[0].annotate(
            event["label"],
            xy=(t_start, price_at_event),
            xytext=(t_start + pd.DateOffset(months=1), price_at_event * 1.03),
            fontsize=8, color=event["color"], fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=event["color"], lw=1)
        )
 
plt.suptitle(
    "TraderEdge Backtest — Did the signal lead the market event?",
    fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig("models/backtest_chart.png", dpi=140, bbox_inches="tight")
plt.show()
print("Saved: models/backtest_chart.png  ← use this in demo slide")

In [ ]:
# %% ─── CELL 10: BACKTEST — Lead time table ───────────────────────────────────
# The key metric for the judge Q&A:
# "How many weeks before the event did TraderEdge flag HIGH risk?"
 
print("=" * 60)
print("BACKTEST LEAD TIME SUMMARY")
print("=" * 60)
 
backtest_results = []
 
for event in EVENTS:
    event_start  = pd.Timestamp(event["start"])
    signal_type  = event["signal_type"]
 
    # Skip if event is outside our data range
    if event_start > signals_df["ds"].max():
        print(f"\n{event['label']}: outside data range — skip")
        continue
 
    # Find the first HIGH signal of correct type before the event
    pre_event_highs = signals_df[
        (signals_df["ds"] < event_start) &
        (signals_df["dominant_signal"] == signal_type) &
        (signals_df["intensity"] == "HIGH")
    ]
 
    # Also check MEDIUM signals (weaker but earlier warning)
    pre_event_meds = signals_df[
        (signals_df["ds"] < event_start) &
        (signals_df["dominant_signal"] == signal_type) &
        (signals_df["intensity"] == "MEDIUM")
    ]
 
    print(f"\n{event['label']} ({signal_type})")
    print(f"  Event start: {event_start.date()}")
 
    if len(pre_event_highs) > 0:
        first_high = pre_event_highs["ds"].max()
        lead_weeks = (event_start - first_high).days // 7
        print(f"  First HIGH signal : {first_high.date()}")
        print(f"  Lead time         : {lead_weeks} weeks  {'✓' if lead_weeks > 0 else '⚠'}")
        backtest_results.append({
            "event": event["label"],
            "signal_type": signal_type,
            "first_high_date": first_high.date(),
            "lead_weeks": lead_weeks
        })
    else:
        print(f"  ⚠ No HIGH {signal_type} signal found before event")
        if len(pre_event_meds) > 0:
            first_med = pre_event_meds["ds"].max()
            lead_weeks = (event_start - first_med).days // 7
            print(f"  First MEDIUM signal: {first_med.date()} ({lead_weeks}w before event)")
        backtest_results.append({
            "event": event["label"],
            "signal_type": signal_type,
            "first_high_date": None,
            "lead_weeks": 0
        })
 
print("\n" + "=" * 60)
print("Presentation framing (use this exact wording in demo):")
for r in backtest_results:
    if r["lead_weeks"] > 0:
        print(f"  '{r['event']}: model flagged {r['signal_type']} risk")
        print(f"   approximately {r['lead_weeks']} weeks before the price event.'")
    else:
        print(f"  '{r['event']}: model did not flag HIGH signal before event.")
        print(f"   Document honestly in limitations slide.'")
print("=" * 60)

In [ ]:
# %% ─── CELL 11: BACKTEST — Simulated trader decision impact ─────────────────
# Illustrative P&L simulation: what would a trader have gained/avoided
# by acting on the TraderEdge signal vs doing nothing?
# NOTE: This is illustrative, not a rigorous financial model.
 
print("=" * 60)
print("SIMULATED TRADER DECISION IMPACT (illustrative)")
print("=" * 60)
print("Assumption: trader manages 1 tonne of rice stock per cycle")
print()
 
for event in EVENTS:
    t_start = pd.Timestamp(event["start"])
    t_end   = pd.Timestamp(event["end"])
 
    price_rows_before = master[master["ds"] <= t_start]
    price_rows_after  = master[master["ds"] <= t_end]
 
    if len(price_rows_before) == 0 or len(price_rows_after) == 0:
        continue
 
    p_before = price_rows_before["y"].iloc[-1]
    p_peak   = price_rows_after["y"].iloc[-1]
    pct_chg  = (p_peak - p_before) / p_before * 100
 
    print(f"{event['label']}")
    print(f"  Price at event start : Rp {p_before:,.0f}/kg")
    print(f"  Price at event peak  : Rp {p_peak:,.0f}/kg")
    print(f"  Change               : {pct_chg:+.1f}%")
 
    if event["signal_type"] == "GLUT":
        saving = (p_before - p_peak) * 1000   # per tonne
        print(f"  Trader WITH signal   : Delayed 1t purchase → avoided Rp {abs(saving):,.0f} overpayment")
        print(f"  Trader WITHOUT       : Bought at peak, overpaid by {abs(pct_chg):.1f}%")
    else:
        gain = (p_peak - p_before) * 1000
        print(f"  Trader WITH signal   : Pre-stocked 1t early → avoided Rp {gain:,.0f} extra cost")
        print(f"  Trader WITHOUT       : Had to restock at peak price")
    print()

In [ ]:
# %% ─── CELL 12: Cross-validation (optional but impressive for judges) ────────
# Prophet has built-in cross-validation. Run this if time allows.
# Uses expanding window: train on earlier data, test on later data.
# Shows the model is robust across different time periods, not just holdout.
 
from prophet.diagnostics import cross_validation, performance_metrics
 
print("Running Prophet cross-validation...")
print("(This may take 1–2 minutes — skip if short on time)")
print()
 
try:
    # initial = first training window, period = step between cutoffs,
    # horizon = how far ahead to test
    # Adjust based on your data length
    n_months = len(train)
    initial_months = max(24, int(n_months * 0.6))   # at least 24 months initial window
 
    df_cv = cross_validation(
        m,
        initial=f"{initial_months * 30} days",
        period="90 days",      # new cutoff every 3 months
        horizon="90 days",     # forecast 3 months ahead
        parallel="processes"
    )
 
    df_perf = performance_metrics(df_cv)
 
    print("Cross-validation performance metrics:")
    print(df_perf[["horizon", "mae", "mape", "coverage"]].to_string(index=False))
    print()
    print(f"Mean directional coverage (80% CI): {df_perf['coverage'].mean():.1%}")
    print(f"Mean MAPE across all horizons:       {df_perf['mape'].mean():.1%}")
    print()
    print("coverage ≈ 0.80 = CI is correctly calibrated (as expected)")
    print("coverage < 0.80 = CI is too narrow (overconfident)")
    print("coverage > 0.80 = CI is too wide (conservative)")
 
    df_perf.to_csv("models/cv_metrics.csv", index=False)
    print("Saved: models/cv_metrics.csv")
 
except Exception as e:
    print(f"Cross-validation skipped: {e}")
    print("This is fine — the backtest (Cell 9-10) is the primary validation.")

In [ ]:
# %% ─── CELL 13: Save model ───────────────────────────────────────────────────
 
joblib.dump(m, "models/prophet_model.pkl")
file_size = os.path.getsize("models/prophet_model.pkl") / (1024 * 1024)
print(f"Saved: models/prophet_model.pkl  ({file_size:.1f} MB)")
 
# Verify round-trip — critical check before deploying to Azure
m_test = joblib.load("models/prophet_model.pkl")
test_future = m_test.make_future_dataframe(periods=3, freq="MS")
for col in FEATURE_COLS:
    test_future[col] = 0.0
test_future["harvest_window"] = test_future["ds"].dt.month.isin([3,4,5,7,8,9]).astype(int)
test_fc = m_test.predict(test_future)
 
print(f"Round-trip test: yhat[-1] = Rp {test_fc['yhat'].iloc[-1]:,.0f}/kg  ✓")

In [ ]:
# %% ─── CELL 14: Save model_card.md ──────────────────────────────────────────
 
model_card = f"""# Model Card — prophet_model.pkl
 
## Model overview
- Type: Facebook Prophet (time-series forecasting)
- Task: Supply pressure signal for rice traders, Pasar Induk Malang, Jawa Timur
- Seasonality mode: **additive** (selected based on EDA — seasonal factor range 0.041)
- Interval width: 80% confidence interval
 
## Training data
- Period: {train['ds'].min().date()} → {train['ds'].max().date()}
- Rows: {len(train)} monthly observations
- Source: WFP/HDX Indonesia Food Prices (Jawa Timur rice series)
 
## Features
| Feature | Prior scale | Role |
|---|---|---|
| production_dev_pct | 0.5 | Primary supply signal (IEEE 2018-2023 + BPS 2025-2026) |
| price_mom_3m | 0.5 | 3-month price momentum |
| price_accel | 0.3 | Momentum acceleration |
| harvest_window | 0.4 | East Java harvest season binary |
| rainfall_dev_pct | 0.02 | Supporting only (downweighted) |
 
## Validation results
- Directional accuracy (holdout): {f'{dir_acc:.1%}' if dir_acc is not None else 'n/a (no holdout data)'}
- MAE: {f'Rp {mae:,.0f}/kg' if mae is not None else 'n/a'}
- MAPE: {f'{mape:.1f}%' if mape is not None else 'n/a'}
 
## Backtest results
{chr(10).join([f'- {r["event"]}: lead time = {r["lead_weeks"]} weeks' for r in backtest_results])}
 
## Known limitations
1. Monthly data resolution — sub-monthly timing cannot be detected
2. 2024 production data is gap-filled using 5-year harvest weight average
3. Rainfall feature is annual → monthly distributed (not station-level daily)
4. Confidence intervals span ±10-25% of price level — directional signal only
5. Model calibrated to Jawa Timur — do not apply to other provinces without retraining
6. Seasonality mode changed to additive (EDA finding: seasonal factor range 0.041 < 0.05)
 
## Output interpretation
Post-processed to GLUT/SHORTAGE/NEUTRAL signal + HIGH/MEDIUM/LOW intensity.
Do NOT expose raw yhat as a precise price prediction.
Designed for pasar induk wholesale traders in Malang area only.
 
## Files
- prophet_model.pkl — serialized model (joblib)
- forecast_plot.png — full forecast visualization
- components_plot.png — trend + seasonality decomposition
- backtest_chart.png — hero backtest visualization
"""
 
with open("models/model_card.md", "w", encoding="utf-8") as f:
    f.write(model_card)
print("Saved: models/model_card.md")

In [ ]:
# %% ─── CELL 15: Final checklist ─────────────────────────────────────────────
 
print("=" * 55)
print("MODEL TRAINING — COMPLETION CHECKLIST")
print("=" * 55)
 
checks = [
    (os.path.exists("models/prophet_model.pkl"),    "prophet_model.pkl saved"),
    (os.path.exists("models/model_card.md"),         "model_card.md saved"),
    (os.path.exists("models/backtest_chart.png"),    "backtest_chart.png saved"),
    (os.path.exists("models/forecast_plot.png"),     "forecast_plot.png saved"),
    (os.path.exists("models/components_plot.png"),   "components_plot.png saved"),
    (file_size < 50,                                 f"Model size < 50MB ({file_size:.1f}MB)"),
    (len(backtest_results) > 0,                      "Backtest results generated"),
]
 
if dir_acc is not None:
    checks.append((dir_acc > 0.50, f"Directional accuracy > 50% ({dir_acc:.1%})"))
 
all_pass = True
for passed, label in checks:
    print(f"  {'✓' if passed else '⚠'}  {label}")
    if not passed:
        all_pass = False
 
print()
if all_pass:
    print("✓ ALL CHECKS PASSED")
    print("  Next step: Module 03 — Azure Function deployment")
    print("  Feed 03_FUNCTION.md to your AI coding assistant")
else:
    print("⚠ Some checks failed — review before proceeding to deployment")
print("=" * 55)
